In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries imported successfully")

Libraries imported successfully


In [3]:
DATA_PATH = Path("../data/interim/Mumbai_AQI_Dataset_Updated_2026.xlsx")

df = pd.read_excel(DATA_PATH)

print(f"Dataset loaded successfully")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

Dataset loaded successfully
Rows: 2771
Columns: 9


In [5]:
df.head()

,City,Date,AQI,PM2.5,PM10,NO2,SO2,CO,O3
0,Mumbai,01/01/2019,194,106.70,209.52,161.02,221.16,2.04,184.30
1,Mumbai,02/01/2019,180,99.00,194.40,149.40,205.20,1.89,171.00
2,Mumbai,03/01/2019,267,146.85,288.36,221.61,304.38,2.80,253.65
3,Mumbai,04/01/2019,223,122.65,240.84,185.09,254.22,2.34,211.85
4,Mumbai,05/01/2019,178,97.90,192.24,147.74,202.92,1.87,169.10


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2771 entries, 0 to 2770
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   City    2771 non-null   object 
 1   Date    2771 non-null   object 
 2   AQI     2771 non-null   int64  
 3   PM2.5   2771 non-null   float64
 4   PM10    2771 non-null   float64
 5   NO2     2771 non-null   float64
 6   SO2     2771 non-null   float64
 7   CO      2771 non-null   float64
 8   O3      2771 non-null   float64
dtypes: float64(6), int64(1), object(2)
memory usage: 195.0+ KB


In [7]:
df.columns.tolist()

['City', 'Date', 'AQI', 'PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3']

In [5]:
df.describe()

,AQI,PM2.5,PM10,NO2,SO2,CO,O3
count,2771.000000,2771.000000,2771.000000,2771.000000,2771.000000,2771.000000,2771.000000
mean,107.207145,58.753739,114.648336,88.167319,121.054998,1.119628,101.645644
std,56.825959,30.969665,61.279644,46.610336,64.255156,0.595689,54.532037
min,9.000000,0.260000,0.000000,8.690000,0.000000,0.000000,10.490000
25%,61.000000,33.550000,64.800000,50.530000,68.400000,0.640000,57.950000
50%,91.000000,49.500000,97.200000,74.700000,102.600000,0.950000,85.500000
75%,149.000000,81.400000,157.680000,121.180000,166.440000,1.540000,141.550000
max,381.000000,209.550000,411.480000,316.230000,434.340000,4.000000,361.950000


In [6]:
missing = df.isnull().sum()

missing_summary = pd.DataFrame({
    "Missing Values": missing,
    "Missing Percentage": (missing / len(df) * 100).round(2)
})

missing_summary

,Missing Values,Missing Percentage
City,0,0.0
Date,0,0.0
AQI,0,0.0
PM2.5,0,0.0
PM10,0,0.0
NO2,0,0.0
SO2,0,0.0
CO,0,0.0
O3,0,0.0


In [7]:
duplicate_count = df.duplicated().sum()

print(f"Duplicate rows: {duplicate_count}")

Duplicate rows: 0


In [8]:
df["Date"] = pd.to_datetime(
    df["Date"],
    format="%d/%m/%Y",
    errors="coerce"
)

print("Invalid dates:", df["Date"].isna().sum())
print("Minimum date:", df["Date"].min())
print("Maximum date:", df["Date"].max())

Invalid dates: 0
Minimum date: 2019-01-01 00:00:00
Maximum date: 2026-08-06 00:00:00


In [11]:
numeric_columns = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO2",
    "SO2",
    "CO",
    "O3"
]

df[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
AQI,2771.0,107.207145,56.825959,9.00,61.00,91.00,149.00,381.00
PM2.5,2771.0,58.753739,30.969665,0.26,33.55,49.50,81.40,209.55
PM10,2771.0,114.648336,61.279644,0.00,64.80,97.20,157.68,411.48
NO2,2771.0,88.167319,46.610336,8.69,50.53,74.70,121.18,316.23
SO2,2771.0,121.054998,64.255156,0.00,68.40,102.60,166.44,434.34
CO,2771.0,1.119628,0.595689,0.00,0.64,0.95,1.54,4.00
O3,2771.0,101.645644,54.532037,10.49,57.95,85.50,141.55,361.95


In [12]:
outlier_summary = []

for column in numeric_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]

    outlier_summary.append({
        "Feature": column,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": len(outliers)
    })

outlier_summary = pd.DataFrame(outlier_summary)

outlier_summary

,Feature,Q1,Q3,IQR,Lower Bound,Upper Bound,Outlier Count
0,AQI,61.00,149.00,88.00,-71.000,281.000,3
1,PM2.5,33.55,81.40,47.85,-38.225,153.175,4
2,PM10,64.80,157.68,92.88,-74.520,297.000,6
3,NO2,50.53,121.18,70.65,-55.445,227.155,10
4,SO2,68.40,166.44,98.04,-78.660,313.500,8
5,CO,0.64,1.54,0.90,-0.710,2.890,6
6,O3,57.95,141.55,83.60,-67.450,266.950,3


In [13]:
REPORT_DIR = Path("../reports")
REPORT_DIR.mkdir(exist_ok=True)

outlier_summary.to_csv(
    REPORT_DIR / "outlier_summary.csv",
    index=False
)

missing_summary.to_csv(
    REPORT_DIR / "missing_values_summary.csv"
)

print("Profiling summaries saved.")

Profiling summaries saved.


In [16]:
from ydata_profiling import ProfileReport

profile = ProfileReport(
    df,
    title="Mumbai AQI Dataset - Data Profiling Report",
    explorative=True
)

profile.to_file(
    "../reports/mumbai_aqi_profiling_report.html"
)

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 94.39it/s]


In [9]:
quality_summary = {
    "Rows": len(df),
    "Columns": len(df.columns),
    "Missing Values": int(df.isnull().sum().sum()),
    "Duplicate Rows": int(df.duplicated().sum()),
    "Invalid Dates": int(df["Date"].isna().sum()),
    "Minimum Date": df["Date"].min(),
    "Maximum Date": df["Date"].max()
}

pd.DataFrame(
    quality_summary.items(),
    columns=["Metric", "Value"]
)

,Metric,Value
0,Rows,2771
1,Columns,9
2,Missing Values,0
3,Duplicate Rows,0
4,Invalid Dates,0
5,Minimum Date,2019-01-01 00:00:00
6,Maximum Date,2026-08-06 00:00:00


In [14]:
# Check for negative values in pollutant and AQI columns

negative_summary = {}

for column in numeric_columns:
    negative_summary[column] = int((df[column] < 0).sum())

negative_summary = pd.Series(negative_summary, name="Negative Values")

negative_summary

AQI      0
PM2.5    0
PM10     0
NO2      0
SO2      0
CO       0
O3       0
Name: Negative Values, dtype: int64

In [15]:
# Check unique cities

print("Unique cities:")
print(df["City"].unique())

print("\nNumber of unique cities:")
print(df["City"].nunique())

Unique cities:
['Mumbai']

Number of unique cities:
1
